In [1]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

In [ ]:
# Find the project root
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Make the project root available to Python
sys.path.insert(0, str(PROJECT_ROOT))

# Load dataset
trans_df = pd.read_csv(r"..\..\data\transactions.csv") 

In [ ]:
from src.data_cleaning import inspect_dataset

inspect_dataset(trans_df)


1. FIRST 3 ROWS
  transaction_id transaction_datetime store_id customer_id payment_method  \
0       T0000001  2024-10-01 08:00:01     S021         NaN         M-Pesa   
1       T0000002  2024-10-01 08:06:49     S007      C02722         M-Pesa   
2       T0000003  2024-10-01 08:07:26     S005         NaN           Card   
3       T0000004  2024-10-01 08:07:44     S009         NaN           Cash   
4       T0000005  2024-10-01 08:10:31     S030      C05664         M-Pesa   

    mpesa_ref  n_lines  basket_total_kes  
0  6O7KQJI5CD        3            1095.0  
1  BH7TIO4ONY        2             759.0  
2         NaN        2            1951.0  
3         NaN        5            2852.0  
4  8PR44LYE42        4            2150.0  

2. DATASET SHAPE
Number of rows: 313136
Number of columns: 8

3. COLUMN NAMES
['transaction_id', 'transaction_datetime', 'store_id', 'customer_id', 'payment_method', 'mpesa_ref', 'n_lines', 'basket_total_kes']

4. DUPLICATE ROWS
Number of duplicate rows: 1248



In [ ]:
# 1. HANDLE DUPLICATES

duplicates = trans_df.duplicated().sum()

# Remove duplicates
df_clean = trans_df.drop_duplicates().copy()

without_duplicates = df_clean.duplicated().sum()

print("Duplicate rows before cleaning:", duplicates)
print("Duplicate row after cleaning:", without_duplicates)


Duplicate rows before cleaning: 1248
Duplicate row after cleaning: 0


In [7]:
# 2. HANDLE MISSING VALUES

# Missing customer_id = walk in customers
df_clean["customer_id"] = df_clean["customer_id"].fillna("WALK-IN")

null_cust_ids = df_clean["customer_id"].isnull().sum() 

print("Missing customer id:", null_cust_ids)

# Missing mpesa reference number = non mpesa payment
df_clean["mpesa_ref"] = df_clean["mpesa_ref"].fillna("NON-MPESA")

null_mpesa_ref = df_clean["mpesa_ref"].isnull().sum()

print("Missing mpesa reference:", null_mpesa_ref)

# Missing payment methods = unknown payment method
df_clean["payment_method"] = df_clean["payment_method"].fillna("Unknown")

null_pay_mthd = df_clean["payment_method"].isnull().sum()

print("Missing payment method:", null_pay_mthd)


Missing customer id: 0
Missing mpesa reference: 0
Missing payment method: 0


In [ ]:
# 3. HANDLE INCONSISTENT PAYMENT METHODS

payment_methods = df_clean.groupby("payment_method")["payment_method"].count().sort_values(ascending=False)
print("Grouped payment methods:\n", payment_methods)


Grouped payment methods:
 payment_method
M-Pesa            157157
Cash               68020
Card               50638
Loyalty Points      6444
mpesa               6286
M-PESA              6249
Mpesa               6219
MPESA               6197
Unknown             4678
Name: payment_method, dtype: int64


In [14]:
# Replace M-pesa values

values = {
    "M-PESA": "M-Pesa",
    "mpesa" : "M-Pesa",
    "Mpesa" : "M-Pesa",
    "MPESA" : "M-Pesa",
    "M-pesa": "M-Pesa"

}

df_clean["payment_method"] = df_clean["payment_method"].replace(values)

payment_methods = df_clean.groupby("payment_method")["payment_method"].count().sort_values(ascending=False)
print("Grouped payment methods:\n", payment_methods)

Grouped payment methods:
 payment_method
M-Pesa            182108
Cash               68020
Card               50638
Loyalty Points      6444
Unknown             4678
Name: payment_method, dtype: int64


In [16]:
# 4. CONVERT DATE TO DATETIME

df_clean["transaction_datetime"] = pd.to_datetime(df_clean["transaction_datetime"])

print("Transaction datetime Dtype:", df_clean["transaction_datetime"].dtype)

Transaction datetime Dtype: datetime64[us]


In [17]:
inspect_dataset(df_clean)


1. FIRST 3 ROWS
  transaction_id transaction_datetime store_id customer_id payment_method  \
0       T0000001  2024-10-01 08:00:01     S021     WALK-IN         M-Pesa   
1       T0000002  2024-10-01 08:06:49     S007      C02722         M-Pesa   
2       T0000003  2024-10-01 08:07:26     S005     WALK-IN           Card   
3       T0000004  2024-10-01 08:07:44     S009     WALK-IN           Cash   
4       T0000005  2024-10-01 08:10:31     S030      C05664         M-Pesa   

    mpesa_ref  n_lines  basket_total_kes  
0  6O7KQJI5CD        3            1095.0  
1  BH7TIO4ONY        2             759.0  
2   NON-MPESA        2            1951.0  
3   NON-MPESA        5            2852.0  
4  8PR44LYE42        4            2150.0  

2. DATASET SHAPE
Number of rows: 311888
Number of columns: 8

3. COLUMN NAMES
['transaction_id', 'transaction_datetime', 'store_id', 'customer_id', 'payment_method', 'mpesa_ref', 'n_lines', 'basket_total_kes']

4. DUPLICATE ROWS
Number of duplicate rows: 0

6. 

In [19]:
# EXPORT THE CLEANED DATASET

from pathlib import Path

output_dir = Path("../../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    output_dir / "transactions_clean.csv",
    index=False
)